<a href="https://colab.research.google.com/github/PhiliS03/us-ie-big-data-technologies/blob/master/postblock3_q4_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Part 2
Question 4 : Apache Beam Analytics

**Objective:**  
Combine `users.csv` and `orders.csv` streams using Beam to generate customer purchasing insights.

**Tasks:**  
1. Join the input files as streams using CoGroupByKey.
2. Perform a transformation that determines the average number of orders for female and male customers,
respectively. Output the result as text in the notebook.
3. Duplicate the code of previous question (in a new cell) and split your pipe to produce/emit the total
number or orders processed as an additional text output (i.e. output the number of orders processed as
well the average orders for female and male customers).
4. Generate a graph (using the Beam library) of your pipeline and upload the image file to STEMLearn.

5. Perform a transformation that groups users into age groups [16-26), [26-36), [36-46), [46-56), and
determine the total number of orders placed by customers in each age group.
6. Determine the total number of times that spinach was purchased within the [16-26), [26-36), [36-46),
[46-56) age groups.
7. Write a pipeline which calculates the average number of orders over seven and thirty day rolling windows
(where the period of each window is one day).
8. Modify this pipeline to write the output to a Parquet file.

---


In [36]:
!pip install --upgrade pandas --quiet

In [37]:
!pip install apache-beam[gcp]==2.57.0 graphviz --quiet

1. Join the input files as streams using CoGroupByKey.

In [39]:
import apache_beam as beam
import pandas as pd
from datetime import datetime
# Load datasets
orders_df = pd.read_csv('/content/orders_v.csv', sep=';')
users_df = pd.read_csv('/content/users_v.csv')
orders_data = orders_df.to_dict(orient='records')
users_data = users_df.to_dict(orient='records')
# Convert 'date' to datetime
def to_timestamp(order):
    # Assuming the date column is named 'date_purchased' based on previous cells
    if 'date_purchased' in order and isinstance(order['date_purchased'], str):
        try:
            order['timestamp'] = datetime.strptime(order['date_purchased'], '%Y-%m-%d')
        except ValueError:
            # Handle cases where date format might be different or invalid
            order['timestamp'] = None # Or some other indicator of invalid date
    else:
        order['timestamp'] = None # Handle cases where 'date_purchased' is missing or not a string
    return order
orders_data = list(map(to_timestamp, orders_data))
with beam.Pipeline() as p:
    # Step 1: Create PCollections
    orders = p | 'Create Orders' >> beam.Create(orders_data)
    users = p | 'Create Users' >> beam.Create(users_data)

    # Step 2: Map each dataset to KV: (customer_id, record)
    # Assuming 'customer_id' is equivalent to 'user_id' based on previous cells
    orders_kv = orders | 'Orders to KV' >> beam.Map(lambda o: (o['user_id'], o))
    users_kv = users | 'Users to KV' >> beam.Map(lambda u: (u['user_id'], u))

    # Step 3: Merge using CoGroupByKey
    merged = ({
        'orders': orders_kv,
        'users': users_kv
    }
    | 'CoGroupByKey' >> beam.CoGroupByKey()
    )
    # Step 4: Flatten the merged records
    def flatten_join(kv):
        customer_id, grouped = kv
        orders_list = grouped['orders']
        users_list = grouped['users']
        if not users_list:
            return  # skip if no user info
        user_info = users_list[0]
        for order in orders_list:
            # Merge user info into each order record
            merged_record = order.copy() # Create a copy to avoid modifying original order
            merged_record.update(user_info)
            yield merged_record


    merged_orders = merged | 'Flatten Merged Records' >> beam.FlatMap(flatten_join)

    # Step 5: Example: print merged orders
    merged_orders | 'Print Merged Orders' >> beam.Map(print)

Streaming output truncated to the last 5000 lines.
{'order_no': 1125834, 'user_id': 2350, 'product_list': 'Nopales, Mushrooms, Daikon Radish', 'date_purchased': '2015-04-13', 'timestamp': datetime.datetime(2015, 4, 13, 0, 0), 'name': 'Kimberly Smith', 'gender': 'female', 'age': 19, 'address': 'East Anthony-GA-00646', 'date_joined': '2021/06/20'}
{'order_no': 1128643, 'user_id': 2350, 'product_list': 'Shallots, Fiddlehead', 'date_purchased': '2015-04-24', 'timestamp': datetime.datetime(2015, 4, 24, 0, 0), 'name': 'Kimberly Smith', 'gender': 'female', 'age': 19, 'address': 'East Anthony-GA-00646', 'date_joined': '2021/06/20'}
{'order_no': 1128863, 'user_id': 2350, 'product_list': 'Bok Choy, Celtuce', 'date_purchased': '2015-04-24', 'timestamp': datetime.datetime(2015, 4, 24, 0, 0), 'name': 'Kimberly Smith', 'gender': 'female', 'age': 19, 'address': 'East Anthony-GA-00646', 'date_joined': '2021/06/20'}
{'order_no': 1135880, 'user_id': 2350, 'product_list': 'Kale, Chayote, Turnip', 'date_p

2. Determines the average number of orders for female and male customers,respectively

In [73]:
import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions
import pandas as pd
import sys
# Suppress Jupyter kernel args warnings
beam_args = [arg for arg in sys.argv if not arg.startswith('-f')]
options = PipelineOptions(argv=beam_args)

users_df = pd.read_csv('users_v.csv')
orders_df = pd.read_csv('orders_v.csv', sep=';')

with beam.Pipeline(options=options) as p:
    users_pc = p | 'Users to Dict' >> beam.Create(users_df.to_dict(orient='records'))
    orders_pc = p | 'Orders to Dict' >> beam.Create(orders_df.to_dict(orient='records'))

    users_kv = users_pc | 'Users KV' >> beam.Map(lambda u: (u['user_id'], u))
    orders_kv = orders_pc | 'Orders KV' >> beam.Map(lambda o: (o['user_id'], o))

    merged_orders = (
        {'users': users_kv, 'orders': orders_kv}
        | 'Join Users and Orders' >> beam.CoGroupByKey()
        | 'Flatten Joined Records' >> beam.FlatMap(
            lambda kv: [
                {**u, **o} for u in kv[1]['users'] for o in kv[1]['orders']
            ]
        )
    )
    # Count orders per user
    orders_per_user = (
        merged_orders
        | 'Map User-Gender to 1' >> beam.Map(lambda o: ((o['user_id'], o['gender']), 1))
        | 'Sum Orders per User' >> beam.CombinePerKey(sum)
    )
    user_orders_gender = orders_per_user | 'Map to Gender' >> beam.Map(lambda kv: (kv[0][1], kv[1]))

    def merge_orders_count(elements):
        total_orders = sum(elements)
        num_users = len(elements)
        return total_orders / num_users if num_users > 0 else 0
    avg_per_gender = (
        user_orders_gender
        | 'Group by Gender' >> beam.GroupByKey()
        | 'Compute Average Orders per Gender' >> beam.Map(lambda kv: {'gender': kv[0], 'average_orders': merge_orders_count(kv[1])})
    )

    def print_result(elements):
        for e in elements:
            print(f"Gender: {e['gender']}, Average Orders: {e['average_orders']:.2f}")

    _ = avg_per_gender | 'ToList for Print' >> beam.combiners.ToList() | 'Print Average Orders' >> beam.Map(print_result)

Gender: male, Average Orders: 678.22
Gender: female, Average Orders: 678.52


3. Duplicate the code of previous question (in a new cell) and split your pipe to produce/emit the total
number or orders processed as an additional text output (i.e. output the number of orders processed as
well the average orders for female and male customers).

In [1]:
import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions
import pandas as pd
import sys
beam_args = [arg for arg in sys.argv if not arg.startswith('-f')]
options = PipelineOptions(argv=beam_args)

users_df = pd.read_csv('users_v.csv')
orders_df = pd.read_csv('orders_v.csv', sep=';') # Added sep=';'
with beam.Pipeline(options=options) as p:
    users_pc = p | 'Users to Dict' >> beam.Create(users_df.to_dict(orient='records'))
    orders_pc = p | 'Orders to Dict' >> beam.Create(orders_df.to_dict(orient='records'))

    users_kv = users_pc | 'Users KV' >> beam.Map(lambda u: (u['user_id'], u))
    orders_kv = orders_pc | 'Orders KV' >> beam.Map(lambda o: (o['user_id'], o))
    merged_orders = (
        {'users': users_kv, 'orders': orders_kv}
        | 'Join Users and Orders' >> beam.CoGroupByKey()
        | 'Flatten Joined Records' >> beam.FlatMap(
            lambda kv: [{**u, **o} for u in kv[1]['users'] for o in kv[1]['orders']]
        )
    )
    orders_per_user = (
        merged_orders
        | 'Map User-Gender to 1' >> beam.Map(lambda o: ((o['user_id'], o['gender']), 1))
        | 'Sum Orders per User' >> beam.CombinePerKey(sum)
    )
    user_orders_gender = orders_per_user | 'Map to Gender' >> beam.Map(lambda kv: (kv[0][1], kv[1]))

    def merge_orders_count(elements):
        return sum(elements)/len(elements) if len(elements) > 0 else 0

    avg_per_gender = (
        user_orders_gender
        | 'Group by Gender' >> beam.GroupByKey()
        | 'Compute Avg Orders per Gender' >> beam.Map(lambda kv: {'gender': kv[0], 'average_orders': merge_orders_count(kv[1])})
    )
    # Total number of orders processed
    total_orders = (
        merged_orders
        | 'Map All Orders to 1' >> beam.Map(lambda o: 1)
        | 'Sum Total Orders' >> beam.CombineGlobally(sum)
    )
    def print_avg_gender(elements):
        for e in elements:
            print(f"Gender: {e['gender']}, Average Orders: {e['average_orders']:.2f}")
    def print_total_orders(elements):
        # elements is a list with a single integer
        print(f"Total Orders Processed: {elements[0]}")
    _ = avg_per_gender | 'ToList Avg Gender' >> beam.combiners.ToList() | 'Print Avg Gender' >> beam.Map(print_avg_gender)
    _ = total_orders | 'ToList Total Orders' >> beam.combiners.ToList() | 'Print Total Orders' >> beam.Map(print_total_orders)

Total Orders Processed: 1598909
Gender: male, Average Orders: 678.22
Gender: female, Average Orders: 678.52


4. Generate a graph (using the Beam library) of your pipeline

In [57]:
import apache_beam as beam
from apache_beam.portability.api import beam_runner_api_pb2
from google.protobuf import text_format
from graphviz import Source
import pandas as pd

# Load dataset
df = pd.read_csv('/content/orders_v.csv', sep=';')
orders_data = df.to_dict(orient='records')

with beam.Pipeline() as p:
    orders = p | '1. Create Orders' >> beam.Create(orders_data)

    # Total orders
    total_orders = (
        orders
        | '2. Count Orders' >> beam.combiners.Count.Globally()
        | '3. Print Total Orders' >> beam.Map(lambda x: print(f"Total Orders: {x}"))
    )

    # Average per gender

    # Convert pipeline to proto
    proto = p.to_runner_api()
    proto_text = text_format.MessageToString(proto)

# Use a simple Graphviz DAG manually
dot = """
digraph BeamPipeline {
    rankdir=LR;
    "Create Orders" -> "Count Orders" -> "Print Total Orders";
    # Need to add steps for joining with users and calculating average per gender
}
"""

s = Source(dot)
s.format = 'png'
s.render('/content/beam_pipeline_graph_real', view=True)

Total Orders: 1598909


'/content/beam_pipeline_graph_real.png'

5. Perform a transformation that groups users into age groups [16-26), [26-36), [36-46), [46-56), and
determine the total number of orders placed by customers in each age group.

In [18]:
import apache_beam as beam
import pandas as pd
from apache_beam.options.pipeline_options import PipelineOptions
import sys
beam_args = [arg for arg in sys.argv if not arg.startswith('-f')]
options = PipelineOptions(argv=beam_args)
# Load datasets using pandas first
users_df = pd.read_csv('users_v.csv')
orders_df = pd.read_csv('orders_v.csv', sep=';') # Added sep=';'
# Convert pandas DataFrames to list of dictionaries
users_data = users_df.to_dict(orient='records')
orders_data = orders_df.to_dict(orient='records')
# Define age groups
def get_age_group(age):
    if 16 <= age < 26:
        return '[16-26)'
    elif 26 <= age < 36:
        return '[26-36)'
    elif 36 <= age < 46:
        return '[36-46)'
    elif 46 <= age < 56:
        return '[46-56)'
    else:
        return 'Other'
def flatten_and_filter_join(kv):
    user_id, grouped = kv
    users_list = grouped.get('users', [])
    orders_list = grouped.get('orders', [])
    if users_list and 'age' in users_list[0]:
        user_info = users_list[0]
        for order in orders_list:
            # Merge user info into each order record
            merged_record = order.copy()
            merged_record.update(user_info)
            yield merged_record
with beam.Pipeline(options=options) as p:
    users_pc = p | 'Users to PCollection' >> beam.Create(users_data)
    orders_pc = p | 'Orders to PCollection' >> beam.Create(orders_data)
  # Join users and orders
    users_kv = users_pc | 'Users KV' >> beam.Map(lambda u: (u['user_id'], u))
    orders_kv = orders_pc | 'Orders KV' >> beam.Map(lambda o: (o['user_id'], o))
    merged = (
        {'users': users_kv, 'orders': orders_kv}
       | 'Join User Orders' >> beam.CoGroupByKey()
        | 'Flatten Joined Records' >> beam.FlatMap(flatten_and_filter_join)
    )
    # Group by age group and count total orders
    orders_per_age_group = (
        merged
        | 'Map Age Group Orders' >> beam.Map(lambda o: (get_age_group(o['age']), 1))
        | 'Sum Orders per Age Group' >> beam.CombinePerKey(sum)
    )
    def print_age_group_results(elements):
        print("=== Total Orders per Age Group ===")
        for age_group, count in elements:
            print(f"Age Group: {age_group}, Total Orders: {count}")
    _ = (
        orders_per_age_group
        | 'ToList Age Groups' >> beam.combiners.ToList()
        | 'Print Age Groups' >> beam.Map(print_age_group_results)
    )

=== Total Orders per Age Group ===
Age Group: Other, Total Orders: 641543
Age Group: [26-36), Total Orders: 271404
Age Group: [46-56), Total Orders: 233317
Age Group: [36-46), Total Orders: 262018
Age Group: [16-26), Total Orders: 190627


6. Determine the total number of times that spinach was purchased within the [16-26), [26-36), [36-46),
[46-56) age groups

In [14]:
import apache_beam as beam
import pandas as pd
from apache_beam.options.pipeline_options import PipelineOptions
import sys

beam_args = [arg for arg in sys.argv if not arg.startswith('-f')]
options = PipelineOptions(argv=beam_args)

# Load data using pandas first
users_df = pd.read_csv('users_v.csv')
orders_df = pd.read_csv('orders_v.csv', sep=';')

# Convert pandas DataFrames to list of dictionaries
users_data = users_df.to_dict(orient='records')
orders_data = orders_df.to_dict(orient='records')

def get_age_group(age):
    if 16 <= age < 26:
        return '[16-26)'
    elif 26 <= age < 36:
        return '[26-36)'
    elif 36 <= age < 46:
        return '[36-46)'
    elif 46 <= age < 56:
        return '[46-56)'
    else:
        return 'Other'
def flatten_and_filter_join(kv):
    user_id, grouped = kv
    users_list = grouped.get('users', [])
    orders_list = grouped.get('orders', [])

    if users_list and 'age' in users_list[0]:
        user_info = users_list[0]
        for order in orders_list:
            # Merge user info into each order record
            merged_record = order.copy()
            merged_record.update(user_info)
            yield merged_record

with beam.Pipeline(options=options) as p:
    # Create PCollections from list of dictionaries
    users_pc = p | 'Users Create' >> beam.Create(users_data)
    orders_pc = p | 'Orders Create' >> beam.Create(orders_data)

    # Merge datasets
    users_kv = users_pc | 'User KV' >> beam.Map(lambda u: (u['user_id'], u))
    orders_kv = orders_pc | 'Order KV' >> beam.Map(lambda o: (o['user_id'], o))

    merged = (
        {'users': users_kv, 'orders': orders_kv}
        | 'Join Data' >> beam.CoGroupByKey()
        | 'Flatten and Filter Joined Records' >> beam.FlatMap(flatten_and_filter_join)
    )
    spinach_purchases = (
        merged
        | 'Filter Spinach' >> beam.Filter(lambda x: 'spinach' in x.get('product_list', '').lower()) # Use .get() and check in product_list
        | 'Map Age Group' >> beam.Map(lambda x: (get_age_group(x['age']), 1))
        | 'Count Spinach' >> beam.CombinePerKey(sum)
    )
    def print_spinach(elements):
        for g, c in elements:
            print(f"Age Group: {g}, Spinach Purchases: {c}")

    _ = spinach_purchases | 'ToList Spinach' >> beam.combiners.ToList() | 'Print Spinach' >> beam.Map(print_spinach)

Age Group: Other, Spinach Purchases: 41086
Age Group: [26-36), Spinach Purchases: 17338
Age Group: [16-26), Spinach Purchases: 12295
Age Group: [36-46), Spinach Purchases: 16594
Age Group: [46-56), Spinach Purchases: 15049


7. Calculate 7-day and 30-day rolling average orders (1-day period)


In [25]:
import apache_beam as beam
import datetime
from apache_beam.options.pipeline_options import PipelineOptions
import sys
import pandas as pd

# Suppress Jupyter kernel args warnings and set up options
beam_args = [arg for arg in sys.argv if not arg.startswith('-f')]
options = PipelineOptions(argv=beam_args)

orders_data = orders_df.to_dict(orient='records')

def parse_order_and_add_timestamp(o):
    # Assuming the date column is named 'date_purchased'
    if 'date_purchased' in o and isinstance(o['date_purchased'], str):
        try:
            # Convert date string to datetime object
            dt_object = datetime.datetime.strptime(o['date_purchased'], '%Y-%m-%d')
            # Assign the datetime object as the event timestamp
            return beam.window.TimestampedValue(o, dt_object.timestamp())
        except ValueError:
            # Handle cases where date format might be different or invalid
            print(f"Warning: Could not parse date for order: {o.get('order_no', 'N/A')}")
            return None
    else:
        print(f"Warning: 'date_purchased' not found or not a string for order: {o.get('order_no', 'N/A')}")
        return None # Return None if 'date_purchased' is missing or not a string

# Define a DoFn to print rolled windowed results
class PrintWindowedResults(beam.DoFn):
    def process(self, element, window=beam.DoFn.WindowParam):
        key, value = element
        # Calculate average per day within the rolled window
        if key == '7_day':
            avg_orders = value / 7.0 if value is not None else 0
            print(f"Window: {window}, 7-Day Avg Orders: {avg_orders:.2f}")
        elif key == '30_day':
            avg_orders = value / 30.0 if value is not None else 0
            print(f"Window: {window}, 30-Day Avg Orders: {avg_orders:.2f}")

with beam.Pipeline(options=options) as p:
    # Create PCollection from list of dictionaries
    orders_pc = p | 'Create Orders' >> beam.Create(orders_data)
    timestamped_orders = orders_pc | 'Parse Dates and Add Timestamps' >> beam.Map(parse_order_and_add_timestamp) | 'Filter Invalid Dates' >> beam.Filter(lambda x: x is not None)

    # Apply 7-day rolled window and sum orders
    windowed_7 = (
        timestamped_orders
        | 'Assign 7-Day Windows' >> beam.WindowInto(
            beam.window.SlidingWindows(size=7*24*60*60, period=24*60*60)
        )
        | 'Map 7-day Orders' >> beam.Map(lambda o: ('7_day', 1))
        | 'Sum 7-day Orders' >> beam.CombinePerKey(sum)
    )
    # Apply 30-day rolled window and sum orders
    windowed_30 = (
        timestamped_orders
        | 'Assign 30-Day Windows' >> beam.WindowInto(
            beam.window.SlidingWindows(size=30*24*60*60, period=24*60*60)
        )
        | 'Map 30-day Orders' >> beam.Map(lambda o: ('30_day', 1))
        | 'Sum 30-day Orders' >> beam.CombinePerKey(sum)
    )
    _ = windowed_7 | 'Print 7-Day Avg' >> beam.ParDo(PrintWindowedResults())
    _ = windowed_30 | 'Print 30-Day Avg' >> beam.ParDo(PrintWindowedResults())

Streaming output truncated to the last 5000 lines.
Window: [1039737600.0, 1042329600.0), 30-Day Avg Orders: 197.43
Window: [1039824000.0, 1042416000.0), 30-Day Avg Orders: 193.83
Window: [1346025600.0, 1348617600.0), 30-Day Avg Orders: 143.47
Window: [1345939200.0, 1348531200.0), 30-Day Avg Orders: 143.77
Window: [1345852800.0, 1348444800.0), 30-Day Avg Orders: 144.23
Window: [1345766400.0, 1348358400.0), 30-Day Avg Orders: 147.77
Window: [1345680000.0, 1348272000.0), 30-Day Avg Orders: 150.70
Window: [1345593600.0, 1348185600.0), 30-Day Avg Orders: 155.23
Window: [1345507200.0, 1348099200.0), 30-Day Avg Orders: 174.90
Window: [1345420800.0, 1348012800.0), 30-Day Avg Orders: 175.40
Window: [1345334400.0, 1347926400.0), 30-Day Avg Orders: 175.47
Window: [1345248000.0, 1347840000.0), 30-Day Avg Orders: 180.07
Window: [1345161600.0, 1347753600.0), 30-Day Avg Orders: 179.03
Window: [1345075200.0, 1347667200.0), 30-Day Avg Orders: 174.30
Window: [1344988800.0, 1347580800.0), 30-Day Avg Orde

8. Modify this pipeline to write the output to a Parquet file.

In [ ]:
# Q8: Write 7-day and 30-day rolling averages to a Parquet file
import apache_beam as beam
import datetime
from apache_beam.options.pipeline_options import PipelineOptions
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import glob, os, sys

# Clean kernel args
beam_args = [arg for arg in sys.argv if not arg.startswith('-f')]
options = PipelineOptions(argv=beam_args)

# Load orders_df (assuming orders_v.csv exists and is semicolon-separated)
orders_df = pd.read_csv('/content/orders_v.csv', sep=';')

orders_data = orders_df.to_dict(orient='records')

def parse_order_and_add_timestamp(o):
    if 'date_purchased' in o and isinstance(o['date_purchased'], str):
        try:
            dt = datetime.datetime.strptime(o['date_purchased'], '%Y-%m-%d')
            return beam.window.TimestampedValue(o, dt.timestamp())
        except ValueError:
            return None
    return None

# Convert results to dicts for Parquet
def format_for_parquet(element, window=beam.DoFn.WindowParam):
    key, total = element
    # Convert Beam Timestamp objects to float timestamps before using utcfromtimestamp
    window_start = datetime.datetime.utcfromtimestamp(window.start.timestamp()).strftime('%Y-%m-%d')
    window_end = datetime.datetime.utcfromtimestamp(window.end.timestamp()).strftime('%Y-%m-%d')
    avg = total / 7.0 if key == '7_day' else total / 30.0
    return {
        'window_type': key,
        'window_start': window_start,
        'window_end': window_end,
        'total_orders_in_window': total, # Added total orders to schema
        'average_orders': float(avg)
    }

# Output directory
output_dir = 'parquet_output_q8'
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'rolling_orders')

with beam.Pipeline(options=options) as p:
    orders_pc = p | 'Create Orders' >> beam.Create(orders_data)
    timestamped = (
        orders_pc
        | 'Add Timestamp' >> beam.Map(parse_order_and_add_timestamp)
        | 'Filter Valid Dates' >> beam.Filter(lambda x: x is not None)
    )

    # --- 7-day window ---
    windowed_7_sum = (
        timestamped
        | 'Assign 7-Day Windows' >> beam.WindowInto(
            beam.window.SlidingWindows(size=7*24*60*60, period=24*60*60))
        | 'Map 7-day Orders' >> beam.Map(lambda o: ('7_day', 1))
        # Replace CombinePerKey with GroupByKey and Map for summing
        | 'Group 7-day Orders' >> beam.GroupByKey()
        | 'Sum 7-day Orders' >> beam.Map(lambda kv: (kv[0], sum(kv[1])))
        | 'Format 7-Day' >> beam.ParDo(format_for_parquet)
    )

    # --- 30-day window ---
    windowed_30_sum = (
        timestamped
        | 'Assign 30-Day Windows' >> beam.WindowInto(
            beam.window.SlidingWindows(size=30*24*60*60, period=24*60*60))
        | 'Map 30-day Orders' >> beam.Map(lambda o: ('30_day', 1))
        # Replace CombinePerKey with GroupByKey and Map for summing
        | 'Group 30-day Orders' >> beam.GroupByKey()
        | 'Sum 30-day Orders' >> beam.Map(lambda kv: (kv[0], sum(kv[1])))
        | 'Format 30-Day' >> beam.ParDo(format_for_parquet)
    )

    # Merge both results and write to Parquet
    combined = (windowed_7_sum, windowed_30_sum) | 'Flatten Windows' >> beam.Flatten()
    _ = combined | 'Write Parquet' >> beam.io.WriteToParquet(
        file_path_prefix=output_path,
        schema=pa.schema([
            ('window_type', pa.string()),
            ('window_start', pa.string()),
            ('window_end', pa.string()),
            ('total_orders_in_window', pa.int64()), # Added total orders to schema
            ('avg_orders', pa.float64())
        ]),
        file_name_suffix='.parquet'
    )

# --- Display the saved Parquet output for submission ---
files = glob.glob(f'{output_path}*.parquet')
if files:
    # Read only a subset for display to avoid excessive output for large files
    try:
        # Read multiple files if necessary (Beam can split output into multiple files)
        df_list = [pq.read_table(f).to_pandas() for f in files]
        df = pd.concat(df_list, ignore_index=True)
        print("✅ Parquet file successfully created and read:")
        display(df.head(10)) # Display first 10 rows using display
    except Exception as e:
        print(f"Error reading Parquet file: {e}")
else:
    print("❌ No Parquet file found in output directory.")